In [1]:
import pandas as pd

freq = pd.read_csv("data/freMTPL2freq.csv")
sev = pd.read_csv("data/freMTPL2sev.csv")

print("Policies:", len(freq))
print("Claim records:", len(sev))
print("Distinct policies with claim records:", sev["IDpol"].nunique())

# policies that report claims (ClaimNb > 0) but have no matching row in sev
mismatched = freq[freq["ClaimNb"] > 0]
mismatched = mismatched[~mismatched["IDpol"].isin(sev["IDpol"])]
print("Policies with ClaimNb > 0 but no claim record:", len(mismatched))

Policies: 678013
Claim records: 26639
Distinct policies with claim records: 24950
Policies with ClaimNb > 0 but no claim record: 9116


In [2]:
print(mismatched["VehAge"].describe())
print(mismatched["Region"].value_counts(normalize=True).head())
print(freq["Region"].value_counts(normalize=True).head())

count    9116.000000
mean        4.084467
std         5.075977
min         0.000000
25%         0.000000
50%         2.000000
75%         7.000000
max        42.000000
Name: VehAge, dtype: float64
Region
Centre                         0.279289
Ile-de-France                  0.144800
Provence-Alpes-Cotes-D'Azur    0.095656
Bretagne                       0.085783
Rhone-Alpes                    0.083151
Name: proportion, dtype: float64
Region
Centre                         0.236870
Rhone-Alpes                    0.125001
Provence-Alpes-Cotes-D'Azur    0.116982
Ile-de-France                  0.102935
Bretagne                       0.062126
Name: proportion, dtype: float64


In [3]:
# Merge claims into policies, aggregating claim amount per policy 
# (a policy can have multiple claim records)
claims_agg = sev.groupby("IDpol")["ClaimAmount"].sum().reset_index()
claims_agg.columns = ["IDpol", "TotalClaimAmount"]

df = freq.merge(claims_agg, on="IDpol", how="left")
df["TotalClaimAmount"] = df["TotalClaimAmount"].fillna(0)

# Flag the mismatched policies instead of silently dropping them
df["HasClaimDataIssue"] = (df["ClaimNb"] > 0) & (df["TotalClaimAmount"] == 0)

# Core actuarial metrics
df["ClaimFrequency"] = df["ClaimNb"] / df["Exposure"]          # claims per policy-year
df["SeverityPerClaim"] = df["TotalClaimAmount"] / df["ClaimNb"].replace(0, pd.NA)  # avg cost per claim
df["PurePremium"] = df["TotalClaimAmount"] / df["Exposure"]     # expected cost per policy-year — the core risk metric

# Risk segments by driver age (standard actuarial bands)
df["DrivAgeBand"] = pd.cut(df["DrivAge"], bins=[17,25,35,45,55,65,100],
                             labels=["18-25","26-35","36-45","46-55","56-65","66+"])

print(df[["ClaimFrequency","SeverityPerClaim","PurePremium"]].describe())
print(df["HasClaimDataIssue"].sum(), "policies flagged with the data issue")

       ClaimFrequency   PurePremium
count   678013.000000  6.780130e+05
mean         0.263964  3.832608e+02
std          4.593916  3.682070e+04
min          0.000000  0.000000e+00
25%          0.000000  0.000000e+00
50%          0.000000  0.000000e+00
75%          0.000000  0.000000e+00
max        732.000117  1.852455e+07
9116 policies flagged with the data issue


In [4]:
print(df["Exposure"].describe())
print((df["Exposure"] < 0.05).sum(), "policies with Exposure under 0.05 years (~18 days)")
print(df.sort_values("ClaimFrequency", ascending=False)[["IDpol","ClaimNb","Exposure","ClaimFrequency"]].head(10))

count    678013.000000
mean          0.528750
std           0.364442
min           0.002732
25%           0.180000
50%           0.490000
75%           0.990000
max           2.010000
Name: Exposure, dtype: float64
42386 policies with Exposure under 0.05 years (~18 days)
        IDpol  ClaimNb  Exposure  ClaimFrequency
2838   5896.0        2  0.002732      732.000117
3360   7045.0        2  0.002732      732.000117
4554  10019.0        2  0.002732      732.000117
2697   5616.0        1  0.002732      366.000059
3357   7039.0        1  0.002732      366.000059
2632   5481.0        1  0.002732      366.000059
920    1921.0        1  0.002732      366.000059
914    1908.0        1  0.002732      366.000059
3198   6675.0        1  0.002732      366.000059
5018  11410.0        1  0.002732      366.000059


In [5]:
df["ClaimNb_raw"] = df["ClaimNb"]
df["Exposure_raw"] = df["Exposure"]

df["ClaimNb"] = df["ClaimNb"].clip(upper=4)
df["Exposure"] = df["Exposure"].clip(upper=1)

# recompute the metrics on corrected values
df["ClaimFrequency"] = df["ClaimNb"] / df["Exposure"]
df["PurePremium"] = df["TotalClaimAmount"] / df["Exposure"]

print(df["ClaimFrequency"].describe())
print(df["PurePremium"].describe())
print((df["Exposure_raw"] > 1).sum(), "policies had Exposure capped")
print((df["ClaimNb_raw"] > 4).sum(), "policies had ClaimNb capped")

count    678013.000000
mean          0.263497
std           4.585560
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         732.000117
Name: ClaimFrequency, dtype: float64
count    6.780130e+05
mean     3.832701e+02
std      3.682070e+04
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.852455e+07
Name: PurePremium, dtype: float64
1224 policies had Exposure capped
9 policies had ClaimNb capped


In [6]:
# Reasonable floor: require at least 1 week of exposure (~0.0192 years) 
# for a policy to be included in rate-based metrics
MIN_EXPOSURE = 7/365

df["LowExposureFlag"] = df["Exposure"] < MIN_EXPOSURE
print(df["LowExposureFlag"].sum(), "policies below 1 week exposure")

# Recompute ratios only for policies above the floor; keep raw rows intact
df["ClaimFrequency"] = df["ClaimNb"] / df["Exposure"]
df["PurePremium"] = df["TotalClaimAmount"] / df["Exposure"]

df_reliable = df[~df["LowExposureFlag"]].copy()
print(df_reliable["ClaimFrequency"].describe())
print(df_reliable["PurePremium"].describe())

13603 policies below 1 week exposure
count    664410.000000
mean          0.186212
std           1.639970
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         100.000000
Name: ClaimFrequency, dtype: float64
count    6.644100e+05
mean     3.004374e+02
std      2.807258e+04
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.852455e+07
Name: PurePremium, dtype: float64


In [7]:
print(df_reliable.sort_values("ClaimFrequency", ascending=False)[["IDpol","ClaimNb","Exposure","ClaimFrequency"]].head(5))
print()
print(df_reliable.sort_values("PurePremium", ascending=False)[["IDpol","ClaimNb","Exposure","TotalClaimAmount","PurePremium"]].head(5))
print()
print(sev["ClaimAmount"].describe())
print(sev.sort_values("ClaimAmount", ascending=False).head(5))

            IDpol  ClaimNb  Exposure  ClaimFrequency
2074       4324.0        2      0.02           100.0
144745  1112625.0        2      0.02           100.0
4991      11324.0        2      0.02           100.0
4988      11316.0        2      0.02           100.0
4916      11077.0        2      0.02           100.0

            IDpol  ClaimNb  Exposure  TotalClaimAmount   PurePremium
150027  1120377.0        1      0.22        4075400.56  1.852455e+07
399662  3083780.0        1      0.02         182050.00  9.102500e+06
368669  3025890.0        2      0.09         382955.14  4.255057e+06
270621  2141337.0        1      0.32        1301172.60  4.066164e+06
54317    110846.0        2      0.43        1404185.52  3.265548e+06

count    2.663900e+04
mean     2.278536e+03
std      2.929748e+04
min      1.000000e+00
25%      6.868100e+02
50%      1.172000e+03
75%      1.228080e+03
max      4.075401e+06
Name: ClaimAmount, dtype: float64
         IDpol  ClaimAmount
9756   1120377   4075400.56


In [8]:
MIN_EXPOSURE = 30/365  # ~1 month

df["LowExposureFlag"] = df["Exposure"] < MIN_EXPOSURE
print(df["LowExposureFlag"].sum(), "policies below 1 month exposure")

df_reliable = df[~df["LowExposureFlag"]].copy()
print(df_reliable["ClaimFrequency"].describe())
print(df_reliable["PurePremium"].describe())

# confirm the catastrophic claim policy survives this filter (it should — Exposure was 0.22, still below 0.082... check)
print(df[df["IDpol"]==1120377][["IDpol","Exposure","LowExposureFlag"]])

114793 policies below 1 month exposure
count    563220.000000
mean          0.127633
std           0.740788
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          40.000000
Name: ClaimFrequency, dtype: float64
count    5.632200e+05
mean     2.465110e+02
std      2.696252e+04
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.852455e+07
Name: PurePremium, dtype: float64
            IDpol  Exposure  LowExposureFlag
150027  1120377.0      0.22            False


In [9]:
# Portfolio-level pure premium and frequency, by risk segment — 
# this is how insurers actually report these metrics
segment_summary = df_reliable.groupby("DrivAgeBand", observed=True).agg(
    TotalExposure=("Exposure", "sum"),
    TotalClaims=("ClaimNb", "sum"),
    TotalClaimAmount=("TotalClaimAmount", "sum"),
    PolicyCount=("IDpol", "count")
).reset_index()

segment_summary["ClaimFrequency"] = segment_summary["TotalClaims"] / segment_summary["TotalExposure"]
segment_summary["PurePremium"] = segment_summary["TotalClaimAmount"] / segment_summary["TotalExposure"]

print(segment_summary)

  DrivAgeBand  TotalExposure  TotalClaims  TotalClaimAmount  PolicyCount  \
0       18-25       15874.70         2623       11397133.78        31578   
1       26-35       67398.31         6043        9702634.95       117627   
2       36-45       85877.58         7722       11616478.97       139254   
3       46-55       87869.30         8505       11485807.00       136822   
4       56-65       51108.42         4399        6069583.96        77892   
5         66+       43914.36         3969        6173353.70        60047   

   ClaimFrequency  PurePremium  
0        0.165231   717.943254  
1        0.089661   143.959618  
2        0.089919   135.267889  
3        0.096791   130.714675  
4        0.086072   118.758983  
5        0.090380   140.577107  


In [10]:
# Repeat the same segment-aggregation logic for other key risk dimensions
def segment_agg(df, group_col):
    s = df.groupby(group_col, observed=True).agg(
        TotalExposure=("Exposure", "sum"),
        TotalClaims=("ClaimNb", "sum"),
        TotalClaimAmount=("TotalClaimAmount", "sum"),
        PolicyCount=("IDpol", "count")
    ).reset_index()
    s["ClaimFrequency"] = s["TotalClaims"] / s["TotalExposure"]
    s["PurePremium"] = s["TotalClaimAmount"] / s["TotalExposure"]
    return s.sort_values("PurePremium", ascending=False)

by_region = segment_agg(df_reliable, "Region")
by_vehpower = segment_agg(df_reliable, "VehPower")
by_area = segment_agg(df_reliable, "Area")
by_vehbrand = segment_agg(df_reliable, "VehBrand")

print(by_region)
print(by_vehpower)
print(by_area)

                         Region  TotalExposure  TotalClaims  TotalClaimAmount  \
7             Champagne-Ardenne        1177.45          128         481248.34   
8                         Corse        1733.04          216         455536.07   
20                  Rhone-Alpes       44605.67         4685        9605596.49   
19  Provence-Alpes-Cotes-D'Azur       34746.79         3535        6524892.60   
6                        Centre      101902.15         8843       18551679.49   
11                Ile-de-France       29489.46         3485        4176752.68   
5                      Bretagne       27592.65         2593        3766926.41   
9                 Franche-Comte         548.66           47          73856.31   
17                     Picardie        3502.76          382         457802.77   
1                     Aquitaine       13868.77         1224        1806169.55   
12         Languedoc-Roussillon       14099.05         1301        1811924.71   
4                     Bourgo

In [11]:
MIN_POLICY_COUNT = 5000  # exclude segments too small to trust for headline claims

by_region_reliable = by_region[by_region["PolicyCount"] >= MIN_POLICY_COUNT]
by_vehpower_reliable = by_vehpower[by_vehpower["PolicyCount"] >= MIN_POLICY_COUNT]

# Export everything Power BI needs
df.to_csv("data/policies_enriched.csv", index=False)
by_region.to_csv("data/segment_region.csv", index=False)
by_vehpower.to_csv("data/segment_vehpower.csv", index=False)
by_area.to_csv("data/segment_area.csv", index=False)


In [12]:
df_reliable.groupby("VehBrand")["IDpol"].count()

VehBrand
B1     137599
B10     14695
B11     10907
B12    133007
B13     10206
B14      3424
B2     136007
B3      44203
B4      20508
B5      29208
B6      23456
Name: IDpol, dtype: int64